In [1]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# !find /content/drive -name "hotel_bookings.csv"

In [3]:
df = pd.read_csv('/content/drive/MyDrive/Projects 2026/Ancillary Revenue Optimization System for Hotels/hotel_bookings.csv')

In [4]:
df.head(6)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03
5,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [5]:
df.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies', 'meal',
       'country', 'market_segment', 'distribution_channel',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'company', 'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'reservation_status', 'reservation_status_date'],
      dtype='object')

In [6]:
#Create upgrade_score

import numpy as np

df['upgrade_score'] = (
    0.3 * (df['lead_time'] / df['lead_time'].max()) +
    0.2 * (df['adr'] / df['adr'].max()) +
    0.2 * df['is_repeated_guest'] +
    0.1 * (df['total_of_special_requests'] / df['total_of_special_requests'].max()) +
    0.2 * (df['stays_in_week_nights'] / df['stays_in_week_nights'].max())
)

# Adding Noise

df['upgrade'] = (
    df['upgrade_score'] + np.random.normal(0, 0.1, len(df))
) > 0.25

df['upgrade'] = df['upgrade'].astype(int)

# Drop upgrade_score

df.drop('upgrade_score', axis=1, inplace=True)

In [7]:
df.head(6)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,upgrade
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01,1
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01,1
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02,0
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02,0
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03,0
5,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03,0


Data Cleaning

In [8]:
df.isnull().sum()

,0
hotel,0
is_canceled,0
lead_time,0
arrival_date_year,0
arrival_date_month,0
arrival_date_week_number,0
arrival_date_day_of_month,0
stays_in_weekend_nights,0
stays_in_week_nights,0
adults,0


Fill the null values


*   Countries: Fill with 'Unknown'
*   Agent: Fill with 0 (no agent, direct booking)
*   Company: Fill with 0 (Personal, not corporate)




In [9]:
df['country'].fillna('Unknown', inplace=True)

/tmp/ipykernel_7844/2372192134.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['country'].fillna('Unknown', inplace=True)


In [10]:
df['agent'] = df['agent'].fillna(0)

In [11]:
df['company'] = df['company'].fillna(0)

In [12]:
df['children'].fillna(df['children'].median(), inplace=True)

/tmp/ipykernel_7844/2643601525.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['children'].fillna(df['children'].median(), inplace=True)


In [13]:
#Create features

df['is_corporate'] = (df['company'] != 0).astype(int)
df['is_direct_booking'] = (df['agent'] == 0).astype(int)

In [14]:
df.isnull().sum() #Looks good!

,0
hotel,0
is_canceled,0
lead_time,0
arrival_date_year,0
arrival_date_month,0
arrival_date_week_number,0
arrival_date_day_of_month,0
stays_in_weekend_nights,0
stays_in_week_nights,0
adults,0


In [15]:
# 3. Drop high-cardinality columns
df.drop(['agent', 'company', 'is_canceled',
'assigned_room_type','country','reservation_status'], axis=1, inplace=True)

In [16]:
# Formating Reservation Status Date

In [17]:
df['reservation_status_date'] = pd.to_datetime(df['reservation_status_date'])

In [18]:
df['reservation_year'] = df['reservation_status_date'].dt.year
df['reservation_month'] = df['reservation_status_date'].dt.month
df['reservation_day'] = df['reservation_status_date'].dt.day

In [19]:
df.drop(['reservation_status_date'], axis=1, inplace=True)

In [20]:
df = pd.get_dummies(df, drop_first=True)

In [21]:
df.head(3)

,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,...,reserved_room_type_F,reserved_room_type_G,reserved_room_type_H,reserved_room_type_L,reserved_room_type_P,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Group,customer_type_Transient,customer_type_Transient-Party
0,342,2015,27,1,0,0,2,0.0,0,0,...,False,False,False,False,False,False,False,False,True,False
1,737,2015,27,1,0,0,2,0.0,0,0,...,False,False,False,False,False,False,False,False,True,False
2,7,2015,27,1,0,1,1,0.0,0,0,...,False,False,False,False,False,False,False,False,True,False


In [22]:
list(df.columns)

['lead_time',
 'arrival_date_year',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'booking_changes',
 'days_in_waiting_list',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests',
 'upgrade',
 'is_corporate',
 'is_direct_booking',
 'reservation_year',
 'reservation_month',
 'reservation_day',
 'hotel_Resort Hotel',
 'arrival_date_month_August',
 'arrival_date_month_December',
 'arrival_date_month_February',
 'arrival_date_month_January',
 'arrival_date_month_July',
 'arrival_date_month_June',
 'arrival_date_month_March',
 'arrival_date_month_May',
 'arrival_date_month_November',
 'arrival_date_month_October',
 'arrival_date_month_September',
 'meal_FB',
 'meal_HB',
 'meal_SC',
 'meal_Undefined',
 'market_segment_Complementary',
 'market_segment_Corporate',
 'market_segment_Direct',
 'mar

In [23]:
# Processing

In [24]:
#Define X and y

In [25]:
X = df.drop('upgrade', axis=1)
y = df['upgrade']

In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=77
)

In [27]:
# Makes features comparable
# Helps Logistic Regression work better

In [28]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [29]:
X.head(3)

,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,...,reserved_room_type_F,reserved_room_type_G,reserved_room_type_H,reserved_room_type_L,reserved_room_type_P,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Group,customer_type_Transient,customer_type_Transient-Party
0,342,2015,27,1,0,0,2,0.0,0,0,...,False,False,False,False,False,False,False,False,True,False
1,737,2015,27,1,0,0,2,0.0,0,0,...,False,False,False,False,False,False,False,False,True,False
2,7,2015,27,1,0,1,1,0.0,0,0,...,False,False,False,False,False,False,False,False,True,False


In [30]:
y.head(3)

,upgrade
0,1
1,1
2,0


In [31]:
# Train Model now:-

In [46]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [47]:
# Evaluate

In [48]:
from sklearn.metrics import classification_report

y_probs = log_model.predict_proba(X_test)[:, 1]

y_pred = (y_probs > 0.2).astype(int)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.95      0.96     22416
           1       0.35      0.39      0.37      1462

    accuracy                           0.92     23878
   macro avg       0.65      0.67      0.66     23878
weighted avg       0.92      0.92      0.92     23878



Hotel wants MORE revenue

👉 Send offers to more people
👉 Accept some wrong predictions

➡️ Use lower threshold (0.2)

Recall > Precision

Reality of Business

Sending an upgrade offer:

Costs almost nothing (email, notification)
Upside = more revenue

Precision vs Recall:-
🔹 Precision
“When I say upgrade… am I right?”

High precision → fewer wrong offers
Low precision → many wasted offers

🔹 Recall
“Out of all upgrades… how many did I catch?”

High recall → I can catch more opportunities
Low recall → I miss many upgrades. We rather offer to more people.

# Random Forest

In [35]:
from sklearn.ensemble import RandomForestClassifier

In [49]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [50]:
#Get Probabilities

y_probs = rf_model.predict_proba(X_test)[:, 1]

In [51]:
y_pred = (y_probs > 0.20).astype(int)

In [52]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.93      0.94     22416
           1       0.27      0.38      0.31      1462

    accuracy                           0.90     23878
   macro avg       0.61      0.66      0.63     23878
weighted avg       0.92      0.90      0.91     23878



In [53]:
import pandas as pd

feature_importance = pd.Series(rf_model.feature_importances_, index=X.columns)
print(feature_importance.sort_values(ascending=False).head(10))

lead_time                         0.153931
adr                               0.103698
reservation_day                   0.080789
arrival_date_day_of_month         0.074517
is_repeated_guest                 0.062613
arrival_date_week_number          0.057380
stays_in_week_nights              0.046325
reservation_month                 0.041403
previous_bookings_not_canceled    0.037449
total_of_special_requests         0.036289
dtype: float64


In [41]:
# Gradient Boosting

In [54]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier()
gb_model.fit(X_train, y_train)

y_probs = gb_model.predict_proba(X_test)[:, 1]
y_pred = (y_probs > 0.2).astype(int)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.96      0.96     22416
           1       0.35      0.37      0.36      1462

    accuracy                           0.92     23878
   macro avg       0.66      0.66      0.66     23878
weighted avg       0.92      0.92      0.92     23878



In [43]:
# XGBoost

In [55]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(eval_metric='logloss')
xgb_model.fit(X_train, y_train)

y_probs = xgb_model.predict_proba(X_test)[:, 1]
y_pred = (y_probs > 0.2).astype(int)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.95      0.95     22416
           1       0.32      0.36      0.34      1462

    accuracy                           0.91     23878
   macro avg       0.64      0.65      0.65     23878
weighted avg       0.92      0.91      0.92     23878



In [56]:
models = {
    "Logistic Regression": log_model,
    "Random Forest": rf_model,
    "Gradient Boosting": gb_model,
    "XGBoost": xgb_model
}

In [57]:
from sklearn.metrics import classification_report

thresholds = [0.2, 0.25, 0.3]

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Model: {name}")

    y_probs = model.predict_proba(X_test)[:, 1]

    for t in thresholds:
        print(f"\nThreshold: {t}")
        y_pred = (y_probs > t).astype(int)
        print(classification_report(y_test, y_pred))


Model: Logistic Regression

Threshold: 0.2
              precision    recall  f1-score   support

           0       0.96      0.95      0.96     22416
           1       0.35      0.39      0.37      1462

    accuracy                           0.92     23878
   macro avg       0.65      0.67      0.66     23878
weighted avg       0.92      0.92      0.92     23878


Threshold: 0.25
              precision    recall  f1-score   support

           0       0.96      0.97      0.96     22416
           1       0.39      0.33      0.36      1462

    accuracy                           0.93     23878
   macro avg       0.67      0.65      0.66     23878
weighted avg       0.92      0.93      0.92     23878


Threshold: 0.3
              precision    recall  f1-score   support

           0       0.95      0.97      0.96     22416
           1       0.42      0.28      0.34      1462

    accuracy                           0.93     23878
   macro avg       0.69      0.63      0.65     238

I evaluated multiple models including Logistic Regression, Random Forest, Gradient Boosting, and XGBoost. After tuning decision thresholds, Logistic Regression with a 0.2 threshold provided the best recall while maintaining reasonable precision, aligning with the goal of maximizing upgrade opportunities.

| Model                 | Threshold | Recall | Precision |
|----------------------|----------|--------|-----------|
| Logistic Regression  | 0.2      | 0.39   | 0.35      |
| Gradient Boosting    | 0.2      | 0.37   | 0.35      |
| Random Forest        | 0.2      | 0.38   | 0.27      |
| XGBoost              | 0.2      | 0.36   | 0.32      |

**Observation:** Logistic Regression achieved the highest recall while maintaining balanced precision.